# Lab 07 · Reference solution

The polished final implementation of [Lab 07: Retrieval strategies and reranking](../README.md).

A production-shaped retrieval pipeline:

- **Dense** (MiniLM bi-encoder, the Lab 06 baseline).
- **BM25** for lexical / proper-noun matching.
- **Reciprocal Rank Fusion** (k=60) for combining them — score-free, robust across corpora.
- **Cross-encoder reranking** (`ms-marco-MiniLM-L-6-v2`) on the top-30 candidates.
- **MMR** available but off by default (high-redundancy queries only).
- Wired into the Lab 06 agent loop as `search_corpus_v2`.

> ⏱ Read time: ~10 min · Notebook ~21 cells.
> 📖 The lab walks each piece (dense → +BM25 → +RRF → +MMR → +reranker)
> through an eval set of 8 queries with rank-shift tracking. This
> solution ships the composed `search_corpus_v2` directly — read the lab
> to see *how* each layer earns its place.

> 🔒 **Chunker config pinned**: `TARGET_TOKENS=160`, `OVERLAP_TOKENS=32`.
> Same as Lab 06; do not change without re-validating downstream labs.

> ⚠️ **MIN_SIMILARITY=0.0 here**, not 0.30. The cross-encoder produces
> *logits*, not cosines — they can be negative for non-matches. Lab 06's
> floor was against MiniLM cosines; this floor is against rerank logits.
> Different scales, different floors.

## Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
from typing import Any

from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Corpus + chunker (same as Lab 06)

Pinned config so chunk IDs match downstream labs.

In [ ]:
CORPUS_DIR = pathlib.Path("../../06-agentic-rag-from-scratch/corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]


def chunk_text(text: str,
               target_tokens: int = TARGET_TOKENS,
               overlap_tokens: int = OVERLAP_TOKENS) -> list[str]:
    """Same recursive chunker as Lab 06. Config pinned for chunk-ID stability."""
    paragraphs = split_at_paragraphs(text)
    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0
    for para in paragraphs:
        para_tokens = approx_tokens(para)
        if para_tokens > target_tokens:
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0
            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > target_tokens and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue
        if current_tokens + para_tokens > target_tokens and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens
    if current:
        chunks.append("\n\n".join(current))
    if overlap_tokens <= 0 or len(chunks) < 2:
        return chunks
    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        overlap_words = int(overlap_tokens * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip() if tail else chunks[i])
    return overlapped


def first_heading(text: str) -> str:
    for line in text.split("\n"):
        if line.startswith("# "):
            return line[2:].strip()
    return ""


all_chunks: list[dict] = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    title = first_heading(text)
    for i, body in enumerate(chunk_text(text)):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "title": title,
            "text": body,
        })

print(f"Loaded {len({c['doc_id'] for c in all_chunks})} docs, {len(all_chunks)} chunks")


**Sample output:**

```
Loaded 8 docs, 55 chunks
```

## Dense index (MiniLM)

Same bi-encoder as Lab 06.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading bi-encoder...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)
chunk_texts = [c["text"] for c in all_chunks]
dense_embeddings = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)
print(f"Dense embeddings: shape={dense_embeddings.shape}")


def dense_retrieve(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    """Dense retrieval. Returns [(chunk_idx, cosine_score), ...]."""
    q_emb = embedder.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )[0]
    scores = dense_embeddings @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices]


## BM25 index

`rank_bm25.BM25Okapi` with defaults `k1=1.5, b=0.75` (Robertson & Walker
standard). Tokenization must be identical between corpus and query —
that's the most common bug in BM25 wiring.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    """Lowercase + extract word tokens >= 2 chars. Apply identically to corpus and query."""
    return [t for t in re.findall(r"\w+", text.lower()) if len(t) > 1]


tokenized_corpus = [tokenize(c["text"]) for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index over {len(tokenized_corpus)} chunks")


def bm25_retrieve(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    """BM25 retrieval. Returns [(chunk_idx, bm25_score), ...]."""
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices]


## Reciprocal Rank Fusion

The classic hybrid combiner: ignore scores (they're on incompatible
scales), use only ranks. RRF score for a doc is the sum over each
retriever's contribution: `1 / (k + rank)`. `k=60` is the standard
robust value — Cormack et al. 2009, Microsoft Research.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: dict[str, list[tuple[int, float]]],
    k: int = 60,
) -> list[tuple[int, float]]:
    """Fuse multiple ranked lists by RRF. Returns [(chunk_idx, rrf_score), ...] desc."""
    rrf_scores: dict[int, float] = {}
    for ranked in ranked_lists.values():
        for rank, (chunk_idx, _score) in enumerate(ranked, start=1):
            rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_retrieve(query: str, top_k: int = 10, candidate_k: int = 30) -> list[tuple[int, float]]:
    """Dense + BM25 fused via RRF. Returns [(chunk_idx, rrf_score), ...]."""
    dense_results = dense_retrieve(query, top_k=candidate_k)
    bm25_results = bm25_retrieve(query, top_k=candidate_k)
    fused = reciprocal_rank_fusion(
        {"dense": dense_results, "bm25": bm25_results},
        k=60,
    )
    return fused[:top_k]


## MMR — Maximal Marginal Relevance

Diversification, when you want it. For most factual queries you do *not*
want it — MMR's "diversity" pulls weakly-relevant chunks from other
documents into the top-k. Useful when an answer requires synthesis
across documents (multi-hop) and the dense retriever returns multiple
chunks of the same document.

`λ=0.7` is a gentle setting; `0.5` is aggressive; `1.0` reduces to pure
relevance.

In [ ]:
def mmr_rerank(
    query: str,
    candidates: list[tuple[int, float]],
    lambda_: float = 0.7,
    top_k: int = 5,
) -> list[tuple[int, float]]:
    """Diversify a candidate set via MMR. Returns [(chunk_idx, mmr_score), ...]."""
    if not candidates:
        return []

    query_emb = embedder.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )[0]

    cand_indices = [c[0] for c in candidates]
    cand_embs = dense_embeddings[cand_indices]
    rel_scores = cand_embs @ query_emb
    cand_sims = cand_embs @ cand_embs.T

    selected: list[int] = []
    selected_scores: list[float] = []

    while len(selected) < min(top_k, len(candidates)):
        if not selected:
            best_pos = int(np.argmax(rel_scores))
            mmr_score = float(rel_scores[best_pos])
        else:
            remaining = [i for i in range(len(candidates)) if i not in selected]
            max_redundancy = cand_sims[remaining][:, selected].max(axis=1)
            mmr_for_remaining = (
                lambda_ * rel_scores[remaining]
                - (1 - lambda_) * max_redundancy
            )
            best_in_remaining = int(np.argmax(mmr_for_remaining))
            best_pos = remaining[best_in_remaining]
            mmr_score = float(mmr_for_remaining[best_in_remaining])
        selected.append(best_pos)
        selected_scores.append(mmr_score)

    return [(cand_indices[pos], score)
            for pos, score in zip(selected, selected_scores, strict=True)]


## Cross-encoder reranker

The single biggest quality lever, applied to a small candidate set
(top-30) post-hybrid. Bi-encoders (dense retrieval) embed query and doc
*independently*; cross-encoders score *the pair* jointly with full
attention. Much slower per pair → only viable on small candidate sets,
which is why it runs *after* the cheaper hybrid retrieval.

`ms-marco-MiniLM-L-6-v2` is the canonical small reranker — 80 MB,
CPU-friendly, trained on MS MARCO passage ranking.

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
    max_length=512,
)


def cross_encoder_rerank(
    query: str,
    candidates: list[tuple[int, float]],
    top_k: int = 5,
) -> list[tuple[int, float]]:
    """Rerank candidates by cross-encoder logit. Returns [(chunk_idx, rerank_logit), ...]."""
    if not candidates:
        return []
    pairs = [(query, all_chunks[idx]["text"]) for idx, _score in candidates]
    rerank_scores = reranker.predict(
        pairs,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    rescored = list(zip([c[0] for c in candidates], rerank_scores.tolist(), strict=True))
    rescored.sort(key=lambda x: x[1], reverse=True)
    return rescored[:top_k]


## `search_corpus_v2` — the composed pipeline

The four-stage pipeline:

1. **Dense + BM25 retrieve** the top-30 candidates each (cheap).
2. **RRF fuse** the two lists into one ranked candidate set.
3. **Cross-encoder rerank** the top-30 → top-5 (the expensive step, scoped to a small set).
4. **Floor at `MIN_SIMILARITY=0.0`** against rerank logits.

MMR is exposed via `use_mmr=False` default; turn it on for queries you
know to be high-redundancy.

In [ ]:
MIN_SIMILARITY = 0.0  # Cross-encoder logits — different scale from Lab 06's 0.30 cosine


def search_corpus_v2(
    query: str,
    top_k: int = 5,
    candidate_k: int = 30,
    use_mmr: bool = False,
    mmr_lambda: float = 0.7,
) -> dict:
    """Production-grade search: dense + BM25 → RRF → (optional MMR) → rerank → top-k."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}

    dense_results = dense_retrieve(query, top_k=candidate_k)
    bm25_results = bm25_retrieve(query, top_k=candidate_k)
    fused = reciprocal_rank_fusion(
        {"dense": dense_results, "bm25": bm25_results},
        k=60,
    )[:candidate_k]

    if use_mmr:
        fused = mmr_rerank(query, fused, lambda_=mmr_lambda, top_k=candidate_k)

    reranked = cross_encoder_rerank(query, fused, top_k=top_k)

    above_floor = [(idx, score) for idx, score in reranked if score >= MIN_SIMILARITY]
    if not above_floor:
        top_score = reranked[0][1] if reranked else float("-inf")
        return {
            "status": "empty", "query": query,
            "detail": f"no chunks crossed rerank floor of {MIN_SIMILARITY} (top score was {top_score:.3f})",
        }

    dense_scores = dict(dense_results)
    bm25_scores = dict(bm25_results)
    results = []
    for idx, rerank_score in above_floor:
        chunk = all_chunks[idx]
        snippet = chunk["text"][:200].replace("\n", " ")
        if len(chunk["text"]) > 200:
            snippet += "..."
        results.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "snippet": snippet,
            "score": float(rerank_score),
            "retrieval_signals": {
                "dense": dense_scores.get(idx, 0.0),
                "bm25": bm25_scores.get(idx, 0.0),
                "rerank": float(rerank_score),
            },
        })
    return {"status": "ok", "results": results}


# Smoke test
print("search_corpus_v2('ReAct pattern thoughts before tool calls'):")
result = search_corpus_v2("ReAct pattern thoughts before tool calls", top_k=3)
for r in result.get("results", []):
    s = r["retrieval_signals"]
    print(f"  rerank={s['rerank']:6.2f}  dense={s['dense']:.3f}  bm25={s['bm25']:.3f}  [{r['chunk_id']}]")


**Sample output (rerank values will vary; ranks usually stable):**

```
search_corpus_v2('ReAct pattern thoughts before tool calls'):
  rerank=  9.42  dense=0.587  bm25=12.31  [03-react-pattern.md:0]
  rerank=  6.18  dense=0.532  bm25=9.07   [03-react-pattern.md:1]
  rerank=  2.45  dense=0.491  bm25=0.00   [01-agent-loop.md:2]
```

`retrieval_signals` is included for debuggability — when the agent
produces a strange answer, looking at which retrieval signal *would*
have surfaced the right chunk tells you which intervention to invest in.

## Wire into the Lab 06 agent loop

Same agent-loop code as Lab 06's solution, just `search_corpus_v2` in
place of `search_corpus`. The tool *contract* hasn't changed — same
`status: ok/empty/error` envelope — so the loop is unchanged.

In [ ]:
def chat_with_tools(messages: list[dict], tools: list[dict]) -> dict:
    """Provider-agnostic chat client — same as Lab 06."""
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "name": tc.function.name, "arguments": tc.function.arguments}
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in tools
        ]
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools, max_tokens=2048, temperature=0,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in resp.content if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


chunks_by_id = {c["chunk_id"]: c for c in all_chunks}


def read_chunk(chunk_id: str) -> dict:
    if not chunk_id:
        return {"status": "error", "kind": "other", "detail": "empty chunk_id"}
    chunk = chunks_by_id.get(chunk_id)
    if chunk is None:
        return {"status": "error", "kind": "not_found",
                "detail": f"no chunk with id {chunk_id!r}"}
    return {
        "status": "ok",
        "chunk_id": chunk["chunk_id"], "doc_id": chunk["doc_id"],
        "title": chunk["title"], "text": chunk["text"],
    }


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_corpus",
            "description": (
                "Search the corpus by hybrid retrieval (dense + BM25 + cross-encoder "
                "reranking). Returns up to top_k chunks. Phrase queries as 3-8 "
                "specific words."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "3-8 specific words."},
                    "top_k": {"type": "integer", "description": "1-10, default 5."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_chunk",
            "description": "Read full text of one chunk. Pass chunk_id from search_corpus.",
            "parameters": {
                "type": "object",
                "properties": {"chunk_id": {"type": "string"}},
                "required": ["chunk_id"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> dict:
    if name == "search_corpus":
        # Wired to v2
        return search_corpus_v2(query=args["query"], top_k=args.get("top_k", 5))
    if name == "read_chunk":
        return read_chunk(chunk_id=args["chunk_id"])
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


MAX_STEPS = 8

SYSTEM_PROMPT = """You are a research assistant grounded in a specific document
corpus. Answer the user's question only from the corpus.

1. Start with search_corpus using 3-8 specific words.
2. If snippets answer the question, synthesize. Otherwise, read_chunk the most
   relevant 1-2 chunks.
3. Cite the chunks you actually read. When the corpus doesn't contain the
   answer, say so plainly.
"""


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


def run_agent(question: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> dict:
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()
    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")
        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)
        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {(msg['content'] or '')[:120]}...")
            return {
                "answer": msg["content"], "citations": citations, "steps": step,
                "stopped_reason": "answer_with_citations" if citations else "answer_without_read",
            }
        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)
            if ah in seen_actions:
                tool_result = {
                    "status": "error", "kind": "repeated_action",
                    "detail": f"You already called {tc['name']} with these arguments.",
                }
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    args_repr = (str(args)[:80] + "...") if len(str(args)) > 80 else str(args)
                    print(f"  → {tc['name']}({args_repr}) → {tool_result.get('status', '?')}")
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append({
                        "chunk_id": tool_result["chunk_id"],
                        "doc_id": tool_result["doc_id"],
                        "title": tool_result["title"],
                    })
            messages.append({
                "role": "tool", "tool_call_id": tc["id"],
                "content": json.dumps(tool_result)[:4000],
            })
    return {
        "answer": f"[step cap; read {len(citations)} chunk(s)]",
        "citations": citations, "steps": max_steps,
        "stopped_reason": "step_cap",
    }


## Demo

In [ ]:
result = run_agent(
    "What's the wrong-but-confident failure mode in retrieval, and what causes it?"
)
print(f"\n=== Answer ===\n{result['answer']}")
print(f"\n=== Citations ({len(result['citations'])}) ===")
for c in result["citations"]:
    print(f"  • [{c['chunk_id']}] {c['title']}")


**Sample output (rank shifts will vary; pipeline behavior should be stable):**

```
── Step 1 ──
  → search_corpus({'query': 'wrong-but-confident retrieval failure'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:2'}) → ok

── Step 3 ──
  ◆ FINAL: The wrong-but-confident failure happens when retrieval surfaces semantically...

=== Citations (1) ===
  • [04-search-vs-retrieval.md:2] Search vs Retrieval
```

## Production readiness — out of scope here

For deployment: ANN-backed dense index (Chroma/Qdrant) once the corpus
grows past ~50K chunks; reranker batching with `batch_size=` tuning;
caching reranker outputs keyed on `(query, chunk_id)`; LLM-as-judge eval
to compare candidate floor settings; A/B testing different RRF `k` values.

For systematic evaluation across pipelines (dense, dense+BM25, +rerank,
full), see Lab 09.